
# Tech Challenge FIAP — Fase 3
## Preparação do modelo para inferência local

### Objetivo

Preparar o modelo Qwen2.5-1.5B-Instruct personalizado para execução no MacBook Air M1.

O carregamento direto em FP16 utilizando PyTorch MPS apresentou insuficiência de memória.

A estratégia adotada consiste em:

1. Carregar o modelo original no Google Colab.
2. Incorporar os pesos do adaptador LoRA.
3. Exportar o modelo consolidado.
4. Converter o modelo para GGUF.
5. Quantizar o modelo para reduzir o consumo de memória.
6. Utilizar Ollama para inferência local.

O adaptador LoRA original será preservado para garantir a reprodutibilidade do processo de fine-tuning.


In [1]:
!pip install -q -U transformers peft accelerate safetensors

# Remover a versão incompatível do torchao
!pip uninstall -y torchao

In [2]:
import torch
from pathlib import Path
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

drive.mount("/content/drive")

assert torch.cuda.is_available(), "Ative a GPU T4."

base_dir = Path(
    "/content/drive/MyDrive/FIAP/TechChallenge_Fase3"
)

adapter_dir = base_dir / "artifacts/qlora_final"
merged_dir = base_dir / "artifacts/qwen_aurora_merged"

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

assert (adapter_dir / "adapter_config.json").exists()
assert (adapter_dir / "adapter_model.safetensors").exists()

# Carregar o modelo original em FP16, sem quantização
tokenizer = AutoTokenizer.from_pretrained(model_id)

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map={"": 0},
    low_cpu_mem_usage=True
)

# Aplicar nosso adaptador
model = PeftModel.from_pretrained(
    base_model,
    str(adapter_dir),
    is_trainable=False
)

# Incorporar os pesos LoRA ao modelo original
merged_model = model.merge_and_unload(safe_merge=True)

# Salvar o modelo consolidado
merged_dir.mkdir(parents=True, exist_ok=True)

merged_model.save_pretrained(
    str(merged_dir),
    safe_serialization=True,
    max_shard_size="2GB"
)

tokenizer.save_pretrained(str(merged_dir))

print("Modelo consolidado salvo com sucesso!")
print(f"Diretório: {merged_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/2 [00:00<?, ?it/s]

Modelo consolidado salvo com sucesso!
Diretório: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/artifacts/qwen_aurora_merged


In [3]:
files = list(merged_dir.iterdir())

for file in sorted(files):
    size_mb = file.stat().st_size / (1024**2)

    print(f"{file.name}: {size_mb:.2f} MB")

assert (merged_dir / "config.json").exists()
assert list(merged_dir.glob("*.safetensors"))

print("\nExportação validada!")

chat_template.jinja: 0.00 MB
config.json: 0.00 MB
generation_config.json: 0.00 MB
model-00001-of-00002.safetensors: 1899.56 MB
model-00002-of-00002.safetensors: 1044.88 MB
model.safetensors.index.json: 0.03 MB
tokenizer.json: 10.89 MB
tokenizer_config.json: 0.00 MB

Exportação validada!


In [4]:
%cd /content

!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git

# Dependências da conversão, sem reinstalar o PyTorch
!pip install -q gguf sentencepiece "protobuf>=4.21,<5"

/content
Cloning into 'llama.cpp'...
remote: Enumerating objects: 4458, done.
remote: Counting objects: 100% (4458/4458), done.
remote: Compressing objects: 100% (3664/3664), done.
remote: Total 4458 (delta 828), reused 2882 (delta 707), pack-reused 0 (from 0)
Receiving objects: 100% (4458/4458), 37.71 MiB | 14.65 MiB/s, done.
Resolving deltas: 100% (828/828), done.
Updating files: 100% (4085/4085), done.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
proto-plus 1.29.0 requires protobuf<8.0.0,>=6.33.5, but you

In [5]:
import subprocess
from pathlib import Path

merged_dir = Path(
    "/content/drive/MyDrive/FIAP/TechChallenge_Fase3/"
    "artifacts/qwen_aurora_merged"
)

gguf_fp16 = Path("/content/qwen_aurora_f16.gguf")

assert (merged_dir / "config.json").exists()

subprocess.run(
    [
        "python",
        "/content/llama.cpp/convert_hf_to_gguf.py",
        str(merged_dir),
        "--outfile", str(gguf_fp16),
        "--outtype", "f16",
    ],
    check=True,
)

print("Conversão concluída!")
print(f"Arquivo: {gguf_fp16}")
print(f"Tamanho: {gguf_fp16.stat().st_size / 1024**3:.2f} GB")

Conversão concluída!
Arquivo: /content/qwen_aurora_f16.gguf
Tamanho: 2.88 GB


In [6]:
import subprocess

subprocess.run(
    [
        "cmake",
        "-S", "/content/llama.cpp",
        "-B", "/content/llama.cpp/build",
        "-DGGML_CUDA=OFF",
        "-DLLAMA_CURL=OFF",
        "-DCMAKE_BUILD_TYPE=Release",
    ],
    check=True,
)

subprocess.run(
    [
        "cmake",
        "--build", "/content/llama.cpp/build",
        "--target", "llama-quantize",
        "-j", "2",
    ],
    check=True,
)

print("Ferramenta de quantização compilada!")

Ferramenta de quantização compilada!


In [7]:
import subprocess
from pathlib import Path

gguf_fp16 = Path("/content/qwen_aurora_f16.gguf")
gguf_q4 = Path("/content/qwen_aurora_q4_k_m.gguf")

subprocess.run(
    [
        "/content/llama.cpp/build/bin/llama-quantize",
        str(gguf_fp16),
        str(gguf_q4),
        "Q4_K_M",
    ],
    check=True,
)

print("Quantização concluída!")
print(f"Tamanho FP16: {gguf_fp16.stat().st_size / 1024**3:.2f} GB")
print(f"Tamanho Q4_K_M: {gguf_q4.stat().st_size / 1024**3:.2f} GB")

Quantização concluída!
Tamanho FP16: 2.88 GB
Tamanho Q4_K_M: 0.92 GB


In [8]:
import shutil
from pathlib import Path

gguf_q4 = Path("/content/qwen_aurora_q4_k_m.gguf")

output_dir = Path(
    "/content/drive/MyDrive/FIAP/TechChallenge_Fase3/"
    "artifacts/gguf"
)

output_dir.mkdir(parents=True, exist_ok=True)

destination = output_dir / gguf_q4.name

shutil.copy2(gguf_q4, destination)

assert destination.exists()
assert destination.stat().st_size == gguf_q4.stat().st_size

print("GGUF salvo no Google Drive!")
print(f"Arquivo: {destination}")
print(f"Tamanho: {destination.stat().st_size / 1024**2:.2f} MB")

GGUF salvo no Google Drive!
Arquivo: /content/drive/MyDrive/FIAP/TechChallenge_Fase3/artifacts/gguf/qwen_aurora_q4_k_m.gguf
Tamanho: 940.37 MB
